# JolRekha — Model Training (from saved Drive data)

Ei notebook **shudhu Drive e save kora data** diye model train kore. News collection / Gemini extraction abar chalate hoy na.

**Input (`MyDrive/jolrekha_data/`):**
- `waterlogging_events_dhaka.csv` — Dataset Builder er final output (dorkar)
- `dhaka_waterlogging_hotspots.csv` — area list (thakle use hobe; na thakle events theke banano hobe)

**Output:** `jolrekha_model.json` — app er `assets/model.json` e boshano jabe (format eki).

**Ager model theke ki improve kora hoyeche (v2):**
1. Jei event er date bujha jay nai (`date_imputed`), oigula label theke bad, karon oigula prai ek din shore thake.
2. `class_weight='balanced'` bad, tai probability ekhon asholi (bristi na thakle ~3%, 21% na).
3. Feature set ar regularization (C) leave-one-year-out diye bachai kora hoy, ar **shudhu oi model nei jar shob coefficient positive** (bristi barle risk komte parbe na).
4. Risk level (উচ্চ/মাঝারি) er threshold hardcode na kore data theke ber kora hoy.
5. Aaj er bristi (`rain_d`) must, ar news er ek din deri dhorar jonno "oi din ba porer din" label o try kore.
6. Save er age sanity check: app er hubohu formula diye scenario test. Fail korle save hoy na.

Runtime → Run all.

In [ ]:
# ===== 1. Setup =====
import json, itertools, math, requests, numpy as np, pandas as pd
from pathlib import Path

try:
    from google.colab import drive, files
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/jolrekha_data')
except ImportError:                      # local run: JR_BASE=/path/to/jolrekha_data
    import os
    BASE, files = Path(os.environ.get('JR_BASE', 'jolrekha_data')), None

EVENTS_CSV   = BASE / 'waterlogging_events_dhaka.csv'
HOTSPOTS_CSV = BASE / 'dhaka_waterlogging_hotspots.csv'
OUT_JSON     = BASE / 'jolrekha_model.json'

LAT, LON = 23.76, 90.39                  # must match AppConfig.dhakaLat / dhakaLon in the app
YEARS  = range(2020, 2026)
MONTHS = range(5, 11)                    # May–October
RAIN_CACHE = BASE / f'rain_hourly_{LAT}_{LON}_{min(YEARS)}_{max(YEARS)}.csv'

DROP_IMPUTED_DATES = True                # events whose date was guessed from the publish date
MIN_PLACES_PER_DAY = 1                   # 2 = only days with 2+ places reported (stricter label)
ALL_FEATURES = ['rain_d', 'rain_prev', 'rain_3d', 'max_1h_2d']   # the app computes exactly these
C_GRID = [0.03, 0.1, 0.3, 1.0, 3.0]
REQUIRE_TODAY_RAIN = True                # the model must use today's rain (rain_d), so the forecast counts
MIN_TODAY_EFFECT = 3.0                   # 100 mm today must be >= 3x riskier than no rain
# News often reports waterlogging the day after the storm. 'same_or_next_day' marks day d as positive
# when waterlogging was reported on d or d+1. Both are tried; the better one is kept.
LABEL_OPTIONS = ['same_day', 'same_or_next_day']
HIGH_LIFT, MEDIUM_LIFT = 10, 3            # 'high' area-days flood 10x more often than an average area-day
print('Data folder:', BASE, '| events file found:', EVENTS_CSV.exists())

## 2. Labels: kon din Dhaka te pani jomeche

In [ ]:
events = pd.read_csv(EVENTS_CSV, encoding='utf-8-sig')
events['event_date'] = pd.to_datetime(events['event_date'], errors='coerce')
events = events.dropna(subset=['event_date']).copy()
print('Columns in the events file:', list(events.columns))
AREA_COLS = [c for c in ['area', 'area_canonical', 'location_key', 'area_en', 'area_as_written'] if c in events]
if not AREA_COLS:
    raise SystemExit('No area column found. Rename the area column in the CSV to "area" and run again.')
area = events[AREA_COLS[0]]
for c in AREA_COLS[1:]:                      # fill gaps from the next available column
    area = area.fillna(events[c])
events['area'] = area.astype(str).str.strip()
print('Area taken from:', AREA_COLS)
n_all = len(events)

if DROP_IMPUTED_DATES and 'date_imputed' in events:
    imputed = events['date_imputed'].astype(str).str.lower() == 'true'
    events = events[~imputed].copy()
    print(f'Dropped {imputed.sum()} of {n_all} events with a guessed date')

events = events[events.event_date.dt.year.isin(YEARS) & events.event_date.dt.month.isin(MONTHS)]
places_per_day = events.groupby('event_date')['area'].nunique()
event_days = set(places_per_day[places_per_day >= MIN_PLACES_PER_DAY].index)
print(f'Events used: {len(events)} | waterlogging days: {len(event_days)}')
print('Days per year:', pd.Series([d.year for d in event_days]).value_counts().sort_index().to_dict())

## 3. Bristi (Open-Meteo archive) ar feature
Prothombar download kore Drive e cache kore rakhe, porer bar download lage na.

In [ ]:
def fetch_hourly_rain(start, end):
    r = requests.get('https://archive-api.open-meteo.com/v1/archive', params={
        'latitude': LAT, 'longitude': LON, 'start_date': start, 'end_date': end,
        'hourly': 'precipitation', 'timezone': 'Asia/Dhaka'}, timeout=180)
    r.raise_for_status()
    h = r.json()['hourly']
    return pd.DataFrame({'time': pd.to_datetime(h['time']), 'rain': h['precipitation']})

if RAIN_CACHE.exists():
    hourly = pd.read_csv(RAIN_CACHE, parse_dates=['time'])
else:
    hourly = fetch_hourly_rain(f'{min(YEARS)}-04-25', f'{max(YEARS)}-10-31')
    hourly.to_csv(RAIN_CACHE, index=False)
hourly['rain'] = hourly['rain'].fillna(0)          # the app treats missing hours as 0 too

hourly['date'] = hourly.time.dt.normalize()
daily = (hourly.groupby('date').agg(rain_d=('rain', 'sum'), max_1h_d=('rain', 'max'))
         .reset_index().sort_values('date').reset_index(drop=True))
# same definitions as RiskService.forecast() in the app
daily['rain_prev'] = daily.rain_d.shift(1)
daily['rain_3d']   = daily.rain_d.rolling(3).sum()
daily['max_1h_2d'] = daily.max_1h_d.rolling(2).max()
daily['rain_2d']   = daily.rain_d + daily.rain_prev
daily = daily[daily.date.dt.year.isin(YEARS) & daily.date.dt.month.isin(MONTHS)].dropna().reset_index(drop=True)

next_days = {d - pd.Timedelta(days=1) for d in event_days}
daily['y_same_day'] = daily.date.isin(event_days).astype(int)
daily['y_same_or_next_day'] = (daily.date.isin(event_days) | daily.date.isin(next_days)).astype(int)
daily['y'] = daily['y_same_day']
daily['year'] = daily.date.dt.year
for f in ALL_FEATURES:
    daily[f'log_{f}'] = np.log1p(daily[f])

print(f'Monsoon days: {len(daily)} | waterlogging days: {daily.y.sum()} ({daily.y.mean():.1%})')
print("Median rain on reported days vs the day before (if 'rain_prev' is higher, news lags the storm):")
print(daily.groupby('y')[['rain_d', 'rain_prev', 'rain_2d', 'max_1h_2d']].median().round(1)
      .rename(index={0: 'normal day (median)', 1: 'waterlogging day (median)'}))

## 4. Model bachai: leave-one-year-out
Protiti label × feature combination × C er jonno ekta ekta bochor baire rekhe test kora hoy. Shudhu oi model gula nei jegulo:
- shob coefficient positive,
- aaj er bristi (`rain_d`) use kore,
- aaj 100 mm hole no-rain er ≥3 gun risk dey.

Duita label er tulona korar jonno log-loss skill score (constant prediction er cheye koto bhalo) use kora hoy.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, brier_score_loss

def make_model(C):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000))

def cols(feats):
    return [f'log_{f}' for f in feats]

def loyo_predictions(feats, C, y_all):
    oof = np.full(len(daily), np.nan)
    for yr in YEARS:
        tr, te = (daily.year != yr).values, (daily.year == yr).values
        if y_all[tr].sum() < 2 or te.sum() == 0:
            continue
        m = make_model(C).fit(daily.loc[tr, cols(feats)], y_all[tr])
        oof[te] = m.predict_proba(daily.loc[te, cols(feats)])[:, 1]
    return oof

def today_effect(model, feats):
    def p(today):
        f = {'rain_d': today, 'rain_prev': 0, 'rain_3d': today, 'max_1h_2d': today / 4}
        return model.predict_proba(pd.DataFrame([{f'log_{k}': np.log1p(f[k]) for k in feats}]))[0, 1]
    return p(100) / max(p(0), 1e-12)

rows = []
for label in LABEL_OPTIONS:
    y_all = daily[f'y_{label}'].values
    ll_const = log_loss(y_all, np.full(len(y_all), y_all.mean()))
    for k in range(1, len(ALL_FEATURES) + 1):
        for feats in itertools.combinations(ALL_FEATURES, k):
            for C in C_GRID:
                oof = loyo_predictions(feats, C, y_all)
                ok = ~np.isnan(oof)
                y = y_all[ok]
                full = make_model(C).fit(daily[cols(feats)], y_all)
                ll = log_loss(y, oof[ok], labels=[0, 1])
                rows.append({'label': label, 'features': feats, 'C': C,
                             'skill': 1 - ll / ll_const, 'log_loss': ll,
                             'roc_auc': roc_auc_score(y, oof[ok]),
                             'pr_auc': average_precision_score(y, oof[ok]),
                             'monotone': bool((full[-1].coef_[0] > 0).all()),
                             'uses_today': 'rain_d' in feats,
                             'today_x': today_effect(full, feats)})
cand = pd.DataFrame(rows)

for label in LABEL_OPTIONS:
    y_all = daily[f'y_{label}']
    print(f'{label:17} positive days {y_all.sum():3} ({y_all.mean():.1%}) | 2-day rain alone ROC-AUC {roc_auc_score(y_all, daily.rain_2d):.3f}')

ok_mask = cand.monotone & (cand.today_x >= MIN_TODAY_EFFECT)
if REQUIRE_TODAY_RAIN:
    ok_mask &= cand.uses_today
good = cand[ok_mask].assign(n=lambda d: d.features.map(len)).sort_values(['skill', 'n'], ascending=[False, True])
print(f'\n{len(cand) - len(good)} of {len(cand)} candidates rejected '
      f'(negative coefficient, no today rain, or 100 mm < {MIN_TODAY_EFFECT}x no rain).')
if good.empty:
    raise SystemExit('No model passed the rules. Try MIN_PLACES_PER_DAY = 1, set REQUIRE_TODAY_RAIN = False, '
                     'or lower MIN_TODAY_EFFECT, then run again.')
print('Best allowed models (leave-one-year-out; skill > 0 means better than a constant guess):')
print(good.head(8).drop(columns=['n', 'monotone', 'uses_today']).round(3).to_string(index=False))
print('\nBest of all candidates, without the rules (for comparison):')
print(cand.sort_values('skill', ascending=False).head(3).drop(columns=['uses_today']).round(3).to_string(index=False))

best = good.iloc[0]
LABEL, FEATURES, C_BEST = best.label, list(best.features), float(best.C)
daily['y'] = daily[f'y_{LABEL}']
print(f'\nChosen: label={LABEL}, features={FEATURES}, C={C_BEST} (100 mm today = {best.today_x:.1f}x no rain)')

## 5. Final model, 2024–2025 test ar chart

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score

X = cols(FEATURES)
train, test = daily[daily.year <= 2023], daily[daily.year >= 2024]
m_split = make_model(C_BEST).fit(train[X], train.y)
p_test = m_split.predict_proba(test[X])[:, 1]

def best_threshold(y, score, grid):
    f1s = [f1_score(y, score >= t, zero_division=0) for t in grid]
    return grid[int(np.argmax(f1s))]

t_model = best_threshold(train.y, m_split.predict_proba(train[X])[:, 1], np.linspace(0.01, 0.6, 60))
t_rain = best_threshold(train.y, train.rain_2d.values, np.arange(5, 150, 1.0))

def report(name, y, score, pred):
    return {'model': name, 'ROC-AUC': roc_auc_score(y, score), 'PR-AUC': average_precision_score(y, score),
            'precision': precision_score(y, pred, zero_division=0), 'recall': recall_score(y, pred, zero_division=0),
            'F1': f1_score(y, pred, zero_division=0)}

results = pd.DataFrame([
    report(f'Baseline: rain_2d > {t_rain:.0f} mm', test.y, test.rain_2d, test.rain_2d >= t_rain),
    report('JolRekha logistic model', test.y, p_test, p_test >= t_model),
]).set_index('model').round(3)
print(f'Train 2020–2023: {train.y.sum()} waterlogging days | Test 2024–2025: {test.y.sum()}')
print(results)

final_model = make_model(C_BEST).fit(daily[X], daily.y)
daily['p_city'] = final_model.predict_proba(daily[X])[:, 1]
coef = final_model[-1].coef_[0]
print('\nCoefficients:', {f: round(float(c), 3) for f, c in zip(FEATURES, coef)})

# calibration check: predicted vs observed, by predicted-probability bin
daily['p_bin'] = pd.cut(daily.p_city, [0, 0.02, 0.05, 0.1, 0.2, 0.4, 1.0])
print('\nCalibration (predicted vs observed):')
print(daily.groupby('p_bin', observed=True).agg(days=('y', 'size'), predicted=('p_city', 'mean'), observed=('y', 'mean')).round(3))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
bins = [0, 5, 10, 20, 35, 50, 75, 100, 150, 1000]
daily['rain_bin'] = pd.cut(daily.rain_2d, bins, right=False)
rate = daily.groupby('rain_bin', observed=True).agg(days=('y', 'size'), rate=('y', 'mean'), pred=('p_city', 'mean'))
labels = [f'{int(b.left)}–{int(b.right) if b.right < 1000 else "+"}' for b in rate.index]
ax[0].bar(labels, rate.rate * 100, color='#2b7bb9', label='observed')
ax[0].plot(labels, rate.pred * 100, 'o-', color='#d9534f', label='model')
ax[0].set_xlabel('Rain over 2 days (mm)'); ax[0].set_ylabel('Days with waterlogging (%)'); ax[0].legend()
ax[0].tick_params(axis='x', rotation=45); ax[0].set_title('Observed vs model, by rainfall')
xs = np.arange(0, 151, 5)
for prev in (0, 30):
    grid = pd.DataFrame({'rain_d': xs, 'rain_prev': prev, 'rain_3d': xs + prev, 'max_1h_2d': np.maximum(xs, prev) / 4})
    ax[1].plot(xs, final_model.predict_proba(np.log1p(grid[FEATURES]).set_axis(X, axis=1))[:, 1] * 100,
               label=f'yesterday {prev} mm')
ax[1].set_xlabel("Today's rain (mm)"); ax[1].set_ylabel('P(Dhaka waterlogs) %'); ax[1].legend()
ax[1].set_title('Model response (must rise with rain)')
plt.tight_layout(); plt.savefig(BASE / 'jolrekha_model_charts.png', dpi=150); plt.show()

## 6. Area risk ar উচ্চ/মাঝারি threshold
`risk = P(Dhaka) × P(area | Dhaka)`, app er moto. Threshold data theke ber kora hoy: historical data te "উচ্চ" area-din e gor er cheye ≥10 gun beshi pani jomeche, "মাঝারি" te ≥3 gun.

In [ ]:
n_days = len(event_days)
def first_col(df, names):
    return next((c for c in names if c in df), None)

place = None
if HOTSPOTS_CSV.exists():
    hot = pd.read_csv(HOTSPOTS_CSV, encoding='utf-8-sig')
    a, la, lo = first_col(hot, ['area', 'area_canonical', 'location_key', 'area_en']), first_col(hot, ['lat', 'latitude']), first_col(hot, ['lon', 'lng', 'longitude'])
    if a and la and lo:
        place = hot.rename(columns={a: 'area', la: 'lat', lo: 'lon'})[['area', 'lat', 'lon']].copy()
        place['area'] = place['area'].astype(str).str.strip()
        print('Areas from', HOTSPOTS_CSV.name)
    else:
        print(f'{HOTSPOTS_CSV.name} has no area/lat/lon columns ({list(hot.columns)}), using the events file')
if place is None:
    if 'lat' not in events or 'lon' not in events:
        raise SystemExit('Need lat/lon: add them to the events CSV or provide dhaka_waterlogging_hotspots.csv')
    place = events.groupby('area').agg(lat=('lat', 'first'), lon=('lon', 'first')).reset_index()
    print('Areas built from the events file')
# event_days always from the cleaned events, so it matches the labels used here
place['event_days'] = place.area.map(events.groupby('area').event_date.nunique()).fillna(0)
place = place.dropna(subset=['lat', 'lon']).drop_duplicates('area')
place['p_place'] = ((place.event_days + 1) / (n_days + 2)).clip(upper=1.0)
place = place.sort_values('p_place', ascending=False).reset_index(drop=True)
print(f'{len(place)} areas; top: {place.area.head(5).tolist()}')

# every (day, area) pair with its risk and whether it was actually reported
reported = set(zip(events.event_date, events.area))
pd_risk = np.outer(daily.p_city.values, place.p_place.values).ravel()
lag = pd.Timedelta(days=1) if LABEL == 'same_or_next_day' else None
pd_hit = np.array([(d, a) in reported or (lag is not None and (d + lag, a) in reported)
                   for d in daily.date for a in place.area], dtype=int)

avg_rate = pd_hit.mean()

def threshold_for(lift):
    precision = lift * avg_rate
    order = np.argsort(-pd_risk)
    prec = np.cumsum(pd_hit[order]) / np.arange(1, len(order) + 1)
    ok = np.where(prec >= precision)[0]
    ok = ok[ok >= 20]                     # ignore the first few pairs (too noisy)
    return float(pd_risk[order][ok.max()]) if len(ok) else None

high = threshold_for(HIGH_LIFT) or float(np.quantile(pd_risk, 0.995))
medium = threshold_for(MEDIUM_LIFT) or float(np.quantile(pd_risk, 0.97))
# keep the coloured bands small, so the map highlights only the riskiest places
high = max(high, float(np.quantile(pd_risk, 0.99)))      # at most 1% of area-days are high
medium = max(medium, float(np.quantile(pd_risk, 0.95)))  # at most 5% are medium or high
medium = min(medium, high * 0.8)
RISK_LEVELS = {'high': round(high, 4), 'medium': round(medium, 4)}

def level(r):
    return 'high' if r >= RISK_LEVELS['high'] else 'medium' if r >= RISK_LEVELS['medium'] else 'low'
lv = np.array([level(r) for r in pd_risk])
summary = pd.DataFrame({'level': lv, 'hit': pd_hit}).groupby('level').agg(area_days=('hit', 'size'), reported=('hit', 'mean'))
summary['lift_vs_average'] = summary.reported / avg_rate
print('Risk levels:', RISK_LEVELS, f'| average area-day rate {avg_rate:.4f}'); print(summary.round(3))

def replay(date, top=8):
    row = daily[daily.date == pd.Timestamp(date)]
    if row.empty:
        print(date, 'not in the data'); return
    p = float(row.p_city.iloc[0])
    rep = set(events[events.event_date == pd.Timestamp(date)].area)
    r = place.assign(risk=p * place.p_place).head(top)
    r['level'] = r.risk.map(level); r['reported'] = r.area.isin(rep).map({True: 'YES', False: ''})
    print(f'\n{date}: rain {row.rain_d.iloc[0]:.0f} mm today, {row.rain_prev.iloc[0]:.0f} mm yesterday -> P(Dhaka) {p:.0%}')
    print(r[['area', 'risk', 'level', 'reported']].round(3).to_string(index=False))

for d in ['2024-05-28', '2024-07-12', '2025-09-22']:
    replay(d)

## 7. Sanity check (app er formula diye) ar save
App `RiskService.pCity()` je bhabe hishab kore, hubohu oi bhabe JSON theke abar hishab kora hoy. Kono check fail korle file save hoy na.

In [ ]:
scaler, clf = final_model.named_steps['standardscaler'], final_model.named_steps['logisticregression']
model_json = {
    'features': FEATURES, 'transform': 'log1p',
    'scaler_mean': scaler.mean_.tolist(), 'scaler_scale': scaler.scale_.tolist(),
    'coef': clf.coef_[0].tolist(), 'intercept': float(clf.intercept_[0]),
    'risk_levels': RISK_LEVELS,
    'places': place[['area', 'lat', 'lon', 'p_place']].round(5).to_dict('records'),
    'trained_on': f'{min(YEARS)}-{max(YEARS)} May-Oct, {len(daily)} days, {int(daily.y.sum())} waterlogging days',
    'label': LABEL,
    'model_selection': {'method': 'leave-one-year-out log-loss skill; positive coefficients; uses today rain', 'C': C_BEST,
                        'loyo_roc_auc': round(float(best.roc_auc), 3), 'loyo_pr_auc': round(float(best.pr_auc), 3)},
    'test_metrics_2024_2025': results.reset_index().to_dict('records'),
}

def app_p_city(m, f):                       # copy of RiskService.pCity() in the app
    z = m['intercept']
    for i, k in enumerate(m['features']):
        z += m['coef'][i] * ((math.log(1 + f.get(k, 0)) - m['scaler_mean'][i]) / m['scaler_scale'][i])
    return 1 / (1 + math.exp(-z))

def scenario(today, yesterday=0, before=0, max1h=None):
    mx = max1h if max1h is not None else max(today, yesterday) / 4
    return {'rain_d': today, 'rain_prev': yesterday, 'rain_3d': today + yesterday + before, 'max_1h_2d': mx}

top_place = model_json['places'][0]
print(f'{"scenario":28} P(Dhaka)  {top_place["area"]}')
checks = {'no rain': scenario(0), 'light 10 mm': scenario(10), 'today 50 mm': scenario(50),
          'today 100 mm': scenario(100), 'yesterday 50 mm': scenario(0, 50), 'today 70, yesterday 25': scenario(70, 25)}
for name, f in checks.items():
    p = app_p_city(model_json, f); r = p * top_place['p_place']
    print(f'{name:28} {p:7.3f}  {r:.3f} {level(r)}')

errors = []
if not all(c > 0 for c in model_json['coef']):
    errors.append('a coefficient is not positive')
ps = [app_p_city(model_json, scenario(x)) for x in range(0, 151, 10)]
if any(b < a - 1e-9 for a, b in zip(ps, ps[1:])):
    errors.append("risk does not rise with today's rain")
if app_p_city(model_json, scenario(0)) > 0.10:
    errors.append('no-rain probability is above 10%')
if app_p_city(model_json, scenario(100)) <= app_p_city(model_json, scenario(0)) * 3:
    errors.append('100 mm today is not clearly riskier than no rain')
if abs(app_p_city(model_json, checks['today 70, yesterday 25'])
       - final_model.predict_proba(pd.DataFrame([{c: np.log1p(checks['today 70, yesterday 25'][f])
                                                   for f, c in zip(FEATURES, X)}]))[0, 1]) > 1e-6:
    errors.append('JSON formula does not match sklearn')

if errors:
    print('\nNOT SAVED:', '; '.join(errors))
else:
    OUT_JSON.write_text(json.dumps(model_json, indent=2, ensure_ascii=False))
    print(f'\nAll checks passed. Saved {OUT_JSON}')
    print("Copy it to the app as assets/model.json")
    if files:
        files.download(str(OUT_JSON))